# GAN: Generative Adversarial Nets (Goodfellow et al., NeurIPS 2014)
### Triển Khai Bằng PyTorch & Thử Nghiệm Đối Chiếu Giá Trị Học Thuật

> **Tác giả:** Ian J. Goodfellow, Jean Pouget-Abadie, Mehdi Mirza, Bing Xu, David Warde-Farley, Sherjil Ozair, Aaron Courville, Yoshua Bengio  
> **Hội nghị:** NeurIPS / NIPS 2014 (Montreal, Canada)  
> **Ý nghĩa lịch sử:** Khởi đầu cuộc cách mạng Generative AI với trò chơi đối kháng Minimax. Loại bỏ hoàn toàn sự phụ thuộc vào ước lượng hàm mật độ tường minh, tạo ra các hình ảnh sắc nét vượt trội.

---

## 🎯 Mục Tiêu Bài Thực Hành
1. **Kiến trúc Generator & Discriminator:** Xây dựng hai mạng nơ-ron đối đầu bằng PyTorch với hàm kích hoạt LeakyReLU(0.2).
2. **Hàm mất mát Non-Saturating Loss:** Huấn luyện Generator bằng $\max_G \log D(G(z))$ để duy trì gradient mạnh mẽ.
3. **Tối ưu hóa luân phiên (Alternating Adam):** Cập nhật $D$ và $G$ theo cơ chế lý thuyết trò chơi hai người.
4. **Academic Value Benchmark:** Đối chiếu định lượng 3 trường phái:
   - *Mô hình 1 (Kỷ nguyên Tiền-GAN: Ước lượng MLE):* Bị bẫy bình quân hóa các mode (Mode-Averaging), sinh mẫu vào khoảng trống ở giữa $\rightarrow$ Mờ nhòe.
   - *Mô hình 2 (GAN Minimax Gốc / Saturating Loss):* Bị triệt tiêu gradient nghiêm trọng khi Discriminator chiếm ưu thế.
   - *Mô hình 3 (Full Non-Saturating GAN - Goodfellow 2014):* Cân bằng Nash tại $D(x) \approx 0.5$, mẫu sinh sắc nét, chuẩn xác.
5. **Trực quan hóa:** Xuất biểu đồ phân bố mẫu sinh và biểu đồ 4 bảng đối chiếu chuyên sâu.

In [ ]:
import os
import time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[✓] Thiết bị tính toán được chọn: {device}")

## PHẦN 1: ĐỊNH NGHĨA KIẾN TRÚC MÔ HÌNH GAN BẰNG PYTORCH

- **Generator:** $z \in \mathbb{R}^2 \to \text{Dense(64)} \to \text{LeakyReLU(0.2)} \to \text{Dense(64)} \to \text{LeakyReLU(0.2)} \to \text{Dense(2)}$.
- **Discriminator:** $x \in \mathbb{R}^2 \to \text{Dense(64)} \to \text{LeakyReLU(0.2)} \to \text{Dense(64)} \to \text{LeakyReLU(0.2)} \to \text{Dense(1)} \to \text{Sigmoid}$.

In [ ]:
from gan_pytorch import Generator, Discriminator

netG = Generator(latent_dim=2, hidden_dim=64, output_dim=2).to(device)
netD = Discriminator(input_dim=2, hidden_dim=64).to(device)
print("[✓] Đã khởi tạo Generator và Discriminator thành công.")

## PHẦN 2: BỘ NẠP DỮ LIỆU ĐA CỤM (MULTIMODAL DATASET)

Phân phối thực tế gồm 4 Mode đối xứng qua các góc phần tư $(\pm 2, \pm 2)$. Khoảng không gian trung tâm $\|x\| < 1.0$ là khoảng trống không có dữ liệu thật.

In [ ]:
from gan_pytorch import generate_multimodal_data

loader, real_data_tensor, centers = generate_multimodal_data(n_samples=1000)
print(f"[*] Số lượng mẫu dữ liệu thật: {len(real_data_tensor)}")
print(f"[*] Tọa độ các tâm Mode: \n{centers}")

## PHẦN 3: ĐỐI CHIẾU GIÁ TRỊ HỌC THUẬT (ACADEMIC VALUE BENCHMARK)

Huấn luyện và so sánh trực diện:
1. **Mô hình 1: Ước lượng MLE (Mode-Averaging)**
2. **Mô hình 2: GAN Minimax Gốc (Saturating Loss)**
3. **Mô hình 3: Full Non-Saturating GAN (Goodfellow 2014)**

In [ ]:
from gan_pytorch import run_academic_value_benchmark

# Chạy thử nghiệm đối chuẩn học thuật
bench_results = run_academic_value_benchmark(device)

## PHẦN 4: TRỰC QUAN HÓA PHÂN BỐ MẪU DỮ LIỆU SINH RA

So sánh phân bố các mẫu sinh ra: Dữ liệu thật (4 Mode) vs MLE (Bị nhòe vào tâm) vs GAN (Bao trùm 4 Mode sắc nét).

In [ ]:
from gan_pytorch import plot_generated_samples_distribution

out_dir = "outputs"
plot_generated_samples_distribution(bench_results, output_dir=out_dir)

# Hiển thị ảnh phân bố mẫu sinh
sample_img_path = os.path.join(out_dir, "gan_generated_samples.png")
if os.path.exists(sample_img_path):
    img_samples = plt.imread(sample_img_path)
    plt.figure(figsize=(18, 5.5))
    plt.imshow(img_samples)
    plt.axis('off')
    plt.title("So Sánh Phân Bố Mẫu Sinh: Dữ Liệu Thật vs MLE vs GAN (Goodfellow 2014)", fontsize=14, fontweight='bold')
    plt.show()

## PHẦN 5: BIỂU ĐỒ ĐỐI CHIẾU HỌC THUẬT CHUYÊN SÂU

Xuất biểu đồ 4 bảng phân tích:
1. **Đường cong mất mát đối nghịch:** Sự cạnh tranh giữa $D$ và $G$.
2. **Cường độ Gradient ban đầu của $G$:** Non-Saturating khắc phục triệt để bão hòa gradient.
3. **Cân bằng Nash:** $D(\text{Real})$ và $D(\text{Fake})$ tiệm cận mức lý thuyết $0.5$.
4. **Tỷ lệ mẫu rơi vào vùng rỗng:** GAN triệt tiêu mẫu rác ở khoảng chân không giữa các cụm.

In [ ]:
from gan_pytorch import plot_comprehensive_academic_comparison

plot_comprehensive_academic_comparison(bench_results, output_dir=out_dir)

# Hiển thị biểu đồ đối chiếu
comp_img_path = os.path.join(out_dir, "gan_academic_comparison.png")
if os.path.exists(comp_img_path):
    img_comp = plt.imread(comp_img_path)
    plt.figure(figsize=(16, 12))
    plt.imshow(img_comp)
    plt.axis('off')
    plt.title("Kết Quả Đối Chiếu Học Thuật Toàn Diện (GAN NeurIPS 2014)", fontsize=14, fontweight='bold')
    plt.show()

## 🎓 TỔNG KẾT & Ý NGHĨA LỊCH SỬ CỦA GAN (NEURIPS 2014)

1. **Đột phá lý thuyết trò chơi:** GAN là công trình đầu tiên đưa lý thuyết trò chơi Minimax vào huấn luyện mạng nơ-ron sâu, mở ra khái niệm học đối kháng (Adversarial Learning).
2. **Giải cứu độ sắc nét:** Khác với Maximum Likelihood bị bẫy bình quân hóa các mode làm mờ ảnh, hàm mất mát của GAN tối ưu hóa độ phân kỳ Jensen-Shannon, ép mô hình phải tạo ra mẫu sắc nét bám sát phân phối thật.
3. **Non-Saturating Loss là chìa khóa:** Mẹo kỹ thuật cực đại hóa $\log D(G(z))$ đã cứu GAN khỏi tình trạng đóng băng gradient ở những epoch đầu tiên.
4. **Nền tảng của kỷ nguyên Deepfakes & StyleGAN:** Dù ngày nay Diffusion Models đang rất phát triển, GAN vẫn là tượng đài bất hủ về tốc độ suy luận thời gian thực và độ sắc nét siêu thực trong xử lý hình ảnh.